# 경주 관광 코스 추천 알고리즘

## 알고리즘 전략

### 1단계: 컨셉 기반 필터링 (Embedding + 유사도)
- 사용자 컨셉을 텍스트로 변환
- 관광지 overview를 임베딩
- 코사인 유사도로 후보군 선정

### 2단계: 경로 최적화 (Greedy TSP)
- 시작점 기준 가장 가까운 곳부터 방문
- 시간 예산 내에서 최대한 많은 장소 방문
- 이동시간 + 체류시간 계산

### 3단계: 1박2일 처리
- 숙박 위치 기준으로 1일차/2일차 분리
- 숙소 근처 관광지 우선 배정

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from math import radians, sin, cos, sqrt, atan2
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 데이터 로드
df_tour = pd.read_csv("_TourSpot__202607062337.csv")
df_accom = pd.read_csv("_AccommodationDetail__202607062338.csv")

print(f"데이터 로드 완료: 관광지 {len(df_tour)}개, 숙박 {len(df_accom)}개")

## 유틸리티 함수

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """두 지점 간 거리 계산 (km)"""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

def estimate_travel_time(distance_km, transport_mode):
    """이동 시간 계산 (분)"""
    speeds = {'walk': 4, 'bicycle': 15, 'public': 30, 'car': 50}
    time_minutes = (distance_km / speeds[transport_mode]) * 60
    if transport_mode == 'public':
        time_minutes += 10
    return time_minutes

def estimate_visit_duration(content_type_id):
    """관광지 타입별 체류 시간 (분)"""
    duration_map = {
        12: 60, 14: 90, 15: 120, 25: 45, 28: 180, 32: 30, 38: 60, 39: 60
    }
    return duration_map.get(content_type_id, 60)

def get_time_budget(duration_type):
    """여행 기간별 시간 예산 (분)"""
    budgets = {'half_day': 240, 'one_day': 480, 'two_days': 960}
    return budgets.get(duration_type, 480)

## 1. 임베딩 모델 로드 및 데이터 전처리

In [ ]:
# 한국어 임베딩 모델 (경량화)
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

print("임베딩 모델 로드 완료")

In [ ]:
# 관광지 텍스트 데이터 준비
df_tour['text_for_embedding'] = df_tour.apply(
    lambda x: f"{x['name']} {x['overview'] if pd.notna(x['overview']) else ''}".strip(),
    axis=1
)

# 좌표 있는 데이터만 사용
df_valid = df_tour[df_tour[['map_x', 'map_y']].notna().all(axis=1)].copy()
df_valid['visit_duration'] = df_valid['content_type_id'].apply(estimate_visit_duration)

print(f"유효 관광지: {len(df_valid)}개")
print(f"텍스트 데이터 샘플:\n{df_valid['text_for_embedding'].iloc[0][:100]}...")

In [ ]:
# 관광지 임베딩 생성 (시간이 걸릴 수 있음)
print("임베딩 생성 중...")
spot_embeddings = model.encode(
    df_valid['text_for_embedding'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"임베딩 완료: shape {spot_embeddings.shape}")

## 2. 추천 시스템 메인 함수

In [ ]:
def build_user_concept_text(companions, theme):
    """
    사용자 선택을 자연어 텍스트로 변환
    
    Args:
        companions: 'alone', 'family', 'couple', 'friends'
        theme: 'scenery', 'history', 'culture', 'food' 등
    """
    concept_map = {
        'companions': {
            'alone': '혼자 여행하기 좋은',
            'family': '가족과 함께 즐기기 좋은',
            'couple': '연인과 데이트하기 좋은',
            'friends': '친구들과 방문하기 좋은'
        },
        'theme': {
            'scenery': '아름다운 경치와 자연을 감상할 수 있는',
            'history': '역사와 문화유산을 체험할 수 있는',
            'culture': '전통 문화와 예술을 즐길 수 있는',
            'food': '맛있는 음식과 먹거리를 즐길 수 있는'
        }
    }
    
    text = f"{concept_map['companions'][companions]} {concept_map['theme'][theme]} 관광지"
    return text

# 테스트
test_concept = build_user_concept_text('couple', 'scenery')
print(f"컨셉 텍스트 예시: {test_concept}")

In [ ]:
def recommend_course(
    duration='one_day',
    companions='alone',
    theme='scenery',
    transport='car',
    start_lat=None,
    start_lon=None,
    top_k_candidates=30
):
    """
    코스 추천 메인 함수
    
    Returns:
        list of dict: 추천 코스 (순서대로)
    """
    # 1. 사용자 컨셉 임베딩
    concept_text = build_user_concept_text(companions, theme)
    concept_embedding = model.encode([concept_text])
    
    # 2. 유사도 계산
    similarities = cosine_similarity(concept_embedding, spot_embeddings)[0]
    df_valid['similarity'] = similarities
    
    # 3. 상위 후보 선정
    df_candidates = df_valid.nlargest(top_k_candidates, 'similarity').copy()
    
    # 4. 시작점 설정
    if start_lat is None or start_lon is None:
        # 기본: 후보 중심점
        start_lat = df_candidates['map_y'].mean()
        start_lon = df_candidates['map_x'].mean()
    
    # 5. 시간 예산
    time_budget = get_time_budget(duration)
    
    # 7. Greedy 경로 생성
    course = greedy_tsp(
        df_candidates,
        start_lat,
        start_lon,
        time_budget,
        transport
    )
    
    return course, concept_text

In [ ]:
def greedy_tsp(df_candidates, start_lat, start_lon, time_budget, transport):
    """
    Greedy TSP 알고리즘으로 경로 생성
    """
    course = []
    remaining = df_candidates.copy()
    current_lat = start_lat
    current_lon = start_lon
    total_time = 0
    
    while len(remaining) > 0 and total_time < time_budget:
        # 현재 위치에서 가장 가까운 장소 찾기
        remaining['distance'] = remaining.apply(
            lambda x: haversine_distance(current_lat, current_lon, x['map_y'], x['map_x']),
            axis=1
        )
        
        # 가장 가까운 곳 선택
        nearest_idx = remaining['distance'].idxmin()
        nearest = remaining.loc[nearest_idx]
        
        # 이동 시간 계산
        travel_time = estimate_travel_time(nearest['distance'], transport)
        visit_time = nearest['visit_duration']
        
        # 시간 예산 체크
        if total_time + travel_time + visit_time > time_budget:
            break
        
        # 코스에 추가
        course.append({
            'order': len(course) + 1,
            'spot_id': nearest['spot_id'],
            'name': nearest['name'],
            'distance_from_prev': nearest['distance'],
            'travel_time': travel_time,
            'visit_duration': visit_time,
            'lat': nearest['map_y'],
            'lon': nearest['map_x'],
            'similarity': nearest['similarity']
        })
        
        # 시간 누적
        total_time += travel_time + visit_time
        
        # 현재 위치 업데이트
        current_lat = nearest['map_y']
        current_lon = nearest['map_x']
        
        # 방문한 곳 제거
        remaining = remaining.drop(nearest_idx)
    
    return course

## 3. 테스트 실행

In [ ]:
# 예시 1: 연인, 하루, 경치, 자차
course1, concept1 = recommend_course(
    duration='one_day',
    companions='couple',
    theme='scenery',
    transport='car'
)

print(f"=== 추천 코스 ===")
print(f"컨셉: {concept1}")
print(f"총 {len(course1)}개 장소\n")

total_time = 0
for spot in course1:
    print(f"{spot['order']}. {spot['name']}")
    print(f"   거리: {spot['distance_from_prev']:.2f}km, 이동: {spot['travel_time']:.0f}분, 체류: {spot['visit_duration']}분")
    print(f"   유사도: {spot['similarity']:.3f}")
    total_time += spot['travel_time'] + spot['visit_duration']
    print()

print(f"총 소요시간: {total_time:.0f}분 ({total_time/60:.1f}시간)")

In [ ]:
# 예시 2: 가족, 반나절, 역사, 도보
course2, concept2 = recommend_course(
    duration='half_day',
    companions='family',
    theme='history',
    transport='walk',
    start_lat=35.83,  # 경주 시내
    start_lon=129.21
)

print(f"=== 추천 코스 2 ===")
print(f"컨셉: {concept2}")
print(f"총 {len(course2)}개 장소\n")

for spot in course2:
    print(f"{spot['order']}. {spot['name']} (유사도: {spot['similarity']:.3f})")

## 4. 시각화

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def visualize_course(course, title="추천 코스"):
    """코스 시각화"""
    if not course:
        print("추천 코스가 없습니다.")
        return
    
    lons = [spot['lon'] for spot in course]
    lats = [spot['lat'] for spot in course]
    
    plt.figure(figsize=(12, 10))
    
    # 경로 그리기
    plt.plot(lons, lats, 'o-', linewidth=2, markersize=10, alpha=0.7)
    
    # 시작점
    plt.scatter(lons[0], lats[0], c='green', s=300, marker='*', 
                edgecolors='black', linewidths=2, label='시작', zorder=5)
    
    # 끝점
    plt.scatter(lons[-1], lats[-1], c='red', s=300, marker='X', 
                edgecolors='black', linewidths=2, label='종료', zorder=5)
    
    # 순서 표시
    for i, spot in enumerate(course):
        plt.annotate(
            f"{i+1}. {spot['name'][:10]}",
            xy=(spot['lon'], spot['lat']),
            xytext=(10, 10),
            textcoords='offset points',
            fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7)
        )
    
    plt.xlabel('경도')
    plt.ylabel('위도')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

visualize_course(course1, f"추천 코스: {concept1}")

## 5. 개선 방향

### 현재 구현 (MVP)
- ✅ 임베딩 기반 컨셉 매칭
- ✅ Greedy TSP로 경로 최적화
- ✅ 시간 예산 관리
- ✅ 제약조건 필터링

### 향후 개선 사항
1. **더 나은 경로 최적화**
   - Greedy → 2-opt, Genetic Algorithm
   - 실제 도로 거리 API 연동 (Google Maps, Kakao 등)

2. **1박2일 처리**
   - 숙소 위치 기준 1일차/2일차 분할
   - 숙소 근처 관광지 우선 배정

3. **더 정교한 필터링**
   - 영업시간 고려
   - 날씨/계절 고려
   - 혼잡도 정보 활용

4. **추가 테마**
   - 포토스팟, 카페투어, 액티비티 등
   - 사용자 리뷰/평점 반영

5. **실시간 추천**
   - 현재 위치 기반 동적 재계산
   - 시간 지연 시 자동 조정

## 6. 저장 및 API 연동 준비

In [ ]:
# 임베딩 저장 (재사용)
np.save('spot_embeddings.npy', spot_embeddings)
df_valid.to_csv('spots_with_embeddings.csv', index=False)

print("임베딩 및 전처리 데이터 저장 완료")

In [ ]:
# API 응답 형식 예시
def format_for_api(course, concept_text):
    """백엔드 API 응답 형식으로 변환"""
    return {
        "concept": concept_text,
        "total_spots": len(course),
        "total_time_minutes": sum(s['travel_time'] + s['visit_duration'] for s in course),
        "course": course
    }

api_response = format_for_api(course1, concept1)
print("API 응답 샘플:")
print(api_response)